# Prozessanalyse: Wo bricht der Bestellprozess?
## Datensatz: Olist Brazilian E-Commerce (Kaggle)
**Autor:** Julio Nzonene  
**Datum:** April 2026  
**Tool:** DuckDB + Python

### Ziel:
Aus den Rohdaten den realen Bestellprozess ableiten und Schwachstellen 
mit konkreten Zahlen belegen - nicht raten, sondern beweisen.

### Prozessschritte:
1. Bestellung eingegangen
2. Genehmigung erteilt
3. Versand vorbereitet
4. Paket an Carrier übergeben
5. Lieferung an Kunde

In [7]:
import duckdb

con = duckdb.connect()

con.execute("""
    CREATE VIEW orders AS 
    SELECT * FROM read_csv_auto('C:/Users/julio/OneDrive/Portfolio/archive (1)/olist_orders_dataset.csv')
""")

print("Verbindung erfolgreich!")

Verbindung erfolgreich!


## Analyse 1: Bestellstatus-Übersicht
Wie viele Bestellungen erreichen welchen Prozessschritt?
Hier sehen wir wo Bestellungen "stecken bleiben".

In [8]:
# Verteilung der Bestellstatus
con.execute("""
    SELECT 
        order_status,
        COUNT(*) AS anzahl,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS prozent
    FROM orders
    GROUP BY order_status
    ORDER BY anzahl DESC
""").fetchdf()

,order_status,anzahl,prozent
0,delivered,96478,97.02
1,shipped,1107,1.11
2,canceled,625,0.63
3,unavailable,609,0.61
4,invoiced,314,0.32
5,processing,301,0.30
6,created,5,0.01
7,approved,2,0.00


## Analyse 2: Wo verliert der Prozess Zeit?
Wir messen die Dauer zwischen jedem Prozessschritt.
Lange Wartezeiten = Bottleneck = Reibung im Prozess.

In [9]:
# Durchschnittliche Dauer zwischen jedem Prozessschritt in Stunden
con.execute("""
    SELECT 
        ROUND(AVG(DATEDIFF('hour', order_purchase_timestamp, order_approved_at)), 1) AS stunden_bis_genehmigung,
        ROUND(AVG(DATEDIFF('hour', order_approved_at, order_delivered_carrier_date)), 1) AS stunden_bis_versand,
        ROUND(AVG(DATEDIFF('hour', order_delivered_carrier_date, order_delivered_customer_date)), 1) AS stunden_bis_lieferung
    FROM orders
    WHERE order_approved_at IS NOT NULL
    AND order_delivered_carrier_date IS NOT NULL
    AND order_delivered_customer_date IS NOT NULL
""").fetchdf()

,stunden_bis_genehmigung,stunden_bis_versand,stunden_bis_lieferung
0,10.3,67.2,223.9


## Analyse 3: Stornierungen - Wann bricht der Prozess ab?
625 Bestellungen wurden storniert.
Wir prüfen in welchem Stadium die Stornierung passiert.

In [11]:
# Stornierungen analysieren
con.execute("""
    SELECT 
        COUNT(*) AS storniert,
        COUNT(order_approved_at) AS hatten_genehmigung,
        COUNT(order_delivered_carrier_date) AS hatten_versand,
        COUNT(order_delivered_customer_date) AS hatten_lieferung
    FROM orders
    WHERE order_status = 'canceled'
""").fetchdf()

,storniert,hatten_genehmigung,hatten_versand,hatten_lieferung
0,625,484,75,6


## Fazit: Prozessschwachstellen

| Prozessschritt | Befund | Auswirkung |
|---|---|---|
| Genehmigung | 10.3 Stunden | Akzeptabel |
| Versand vorbereiten | 67.2 Stunden | Hauptbottleneck |
| Lieferung | 223.9 Stunden | Carrier-abhängig |
| Stornierungen | 484 nach Genehmigung | Direkte Folge des Versand-Bottlenecks |

**Kernaussage:** Der Prozess bricht nicht beim Carrier - er bricht intern.
67 Stunden zwischen Genehmigung und Versand sind vermeidbar und kosten direkt Umsatz.